# Document Parsing and Chunking Explorer

Use this notebook to manually inspect how the repo currently parses and chunks supported local documents.

Supported types covered here: `txt`, `md`, `pdf`, `docx`, `xlsx`.


In [1]:
from __future__ import annotations

from collections import Counter
import json
from pathlib import Path
from pprint import pprint
from typing import Any

from personal_kb.chunking import build_default_chunker_registry
from personal_kb.parsers import build_default_parser_registry


def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "personal_kb").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root(Path.cwd())
PARSER_REGISTRY = build_default_parser_registry()
CHUNKER_REGISTRY = build_default_chunker_registry()
SUPPORTED_EXTENSIONS = ("txt", "md", "pdf", "docx", "xlsx")

print(f"Repo root: {REPO_ROOT}")
print(f"Notebook cwd: {Path.cwd()}")


Repo root: /Users/pelmeshek1706/Desktop/projects/knowledge_agent
Notebook cwd: /Users/pelmeshek1706/Desktop/projects/knowledge_agent/output/jupyter-notebook


## Sample Files

Edit `selected_path` to point at any supported local file. The defaults below use real files already present in this repo.


In [2]:
SAMPLE_PATHS = {
    "txt": REPO_ROOT / "tests" / "fixtures" / "sample_notes.txt",
    "md": REPO_ROOT / "data" / "Product_Requirements_Document_Personal_KB_v0.2.md",
    "pdf": REPO_ROOT / "tests" / "fixtures" / "openwillis_speech_dataset_feature_reference.pdf",
    "docx": REPO_ROOT / "data" / "DRAFT_Tpl_Application_Form_Part_B_HE_EIC_UKRAINE_2025_AIREST_5.docx",
    "xlsx": REPO_ROOT / "tests" / "fixtures" / "sample_budget.xlsx",
}

for extension, path in SAMPLE_PATHS.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{extension:>4}  {status:>7}  {path}")

# Edit this value when you want to inspect a different supported file.
selected_path = str(SAMPLE_PATHS["md"])
selected_path


 txt       OK  /Users/pelmeshek1706/Desktop/projects/knowledge_agent/tests/fixtures/sample_notes.txt
  md       OK  /Users/pelmeshek1706/Desktop/projects/knowledge_agent/data/Product_Requirements_Document_Personal_KB_v0.2.md
 pdf       OK  /Users/pelmeshek1706/Desktop/projects/knowledge_agent/tests/fixtures/openwillis_speech_dataset_feature_reference.pdf
docx       OK  /Users/pelmeshek1706/Desktop/projects/knowledge_agent/data/DRAFT_Tpl_Application_Form_Part_B_HE_EIC_UKRAINE_2025_AIREST_5.docx
xlsx       OK  /Users/pelmeshek1706/Desktop/projects/knowledge_agent/tests/fixtures/sample_budget.xlsx


'/Users/pelmeshek1706/Desktop/projects/knowledge_agent/data/Product_Requirements_Document_Personal_KB_v0.2.md'

## Extension To Registry Map

This cell resolves each supported extension through the repo's existing parser and chunker registries.


In [3]:
def registry_entry(extension: str) -> dict[str, Any]:
    parser = PARSER_REGISTRY.get_parser(extension)
    chunker = CHUNKER_REGISTRY.get_chunker(extension)
    sample_path = SAMPLE_PATHS[extension]
    return {
        "extension": extension,
        "parser": type(parser).__name__,
        "chunker": type(chunker).__name__,
        "sample_path": sample_path.relative_to(REPO_ROOT).as_posix() if sample_path.exists() else None,
    }


EXTENSION_REGISTRY = {
    extension: registry_entry(extension)
    for extension in SUPPORTED_EXTENSIONS
}

pprint(EXTENSION_REGISTRY)


{'docx': {'chunker': 'DocxChunker',
          'extension': 'docx',
          'parser': 'DocxParser',
          'sample_path': 'data/DRAFT_Tpl_Application_Form_Part_B_HE_EIC_UKRAINE_2025_AIREST_5.docx'},
 'md': {'chunker': 'MarkdownChunker',
        'extension': 'md',
        'parser': 'MarkdownParser',
        'sample_path': 'data/Product_Requirements_Document_Personal_KB_v0.2.md'},
 'pdf': {'chunker': 'PdfChunker',
         'extension': 'pdf',
         'parser': 'PdfParser',
         'sample_path': 'tests/fixtures/openwillis_speech_dataset_feature_reference.pdf'},
 'txt': {'chunker': 'TxtChunker',
         'extension': 'txt',
         'parser': 'TxtParser',
         'sample_path': 'tests/fixtures/sample_notes.txt'},
 'xlsx': {'chunker': 'XlsxChunker',
          'extension': 'xlsx',
          'parser': 'XlsxParser',
          'sample_path': 'tests/fixtures/sample_budget.xlsx'}}


## Helpers

These helpers resolve the input path, run the matching parser and chunker, and build compact previews for manual inspection.


In [4]:
def resolve_input_path(path_text: str) -> Path:
    candidate = Path(path_text).expanduser()
    if not candidate.is_absolute():
        candidate = REPO_ROOT / candidate
    return candidate.resolve()


def extension_for(path: Path) -> str:
    extension = path.suffix.lower().lstrip(".")
    if extension not in SUPPORTED_EXTENSIONS:
        supported = ", ".join(SUPPORTED_EXTENSIONS)
        raise ValueError(f"Unsupported extension '{extension}'. Expected one of: {supported}")
    return extension


def source_id_for(path: Path) -> str:
    try:
        return path.relative_to(REPO_ROOT).as_posix()
    except ValueError:
        return path.as_posix()


def preview_text(text: str, limit: int = 160) -> str:
    compact = " ".join(text.split())
    if len(compact) <= limit:
        return compact
    return compact[: limit - 3] + "..."


def summarize_block(block: dict[str, Any]) -> dict[str, Any]:
    return {
        "type": block.get("type"),
        "heading_path": block.get("heading_path"),
        "page_number": block.get("page_number"),
        "sheet_name": block.get("sheet_name"),
        "cell_range": block.get("cell_range"),
        "source_ref": block.get("source_ref"),
        "text_preview": preview_text(str(block.get("text", ""))),
    }


def summarize_chunk(chunk: Any) -> dict[str, Any]:
    return {
        "chunk_id": chunk.chunk_id,
        "chunk_index": chunk.chunk_index,
        "char_count": chunk.char_count,
        "source_ref": chunk.source_ref.model_dump(mode="json"),
        "text_preview": preview_text(chunk.text),
    }


def collect_stats(parsed: Any, chunks: list[Any]) -> dict[str, Any]:
    chunk_lengths = [len(chunk.text) for chunk in chunks]
    source_ref_keys = [
        json.dumps(chunk.source_ref.model_dump(mode="json"), sort_keys=True)
        for chunk in chunks
    ]
    return {
        "file_path": parsed.file_path,
        "file_extension": parsed.file_extension,
        "parser": type(PARSER_REGISTRY.get_parser(parsed.file_extension)).__name__,
        "chunker": type(CHUNKER_REGISTRY.get_chunker(parsed.file_extension)).__name__,
        "raw_text_chars": len(parsed.raw_text),
        "structured_block_count": len(parsed.structured_blocks),
        "block_type_counts": dict(Counter(str(block.get("type", "unknown")) for block in parsed.structured_blocks)),
        "chunk_count": len(chunks),
        "chunk_char_min": min(chunk_lengths) if chunk_lengths else 0,
        "chunk_char_max": max(chunk_lengths) if chunk_lengths else 0,
        "chunk_char_avg": round(sum(chunk_lengths) / len(chunk_lengths), 1) if chunk_lengths else 0,
        "unique_chunk_source_refs": len(set(source_ref_keys)),
    }


def inspect_document(path_text: str) -> dict[str, Any]:
    path = resolve_input_path(path_text)
    extension = extension_for(path)
    parser = PARSER_REGISTRY.get_parser(extension)
    chunker = CHUNKER_REGISTRY.get_chunker(extension)
    source_id = source_id_for(path)
    parsed = parser.parse(path, source_id=source_id)
    chunks = chunker.chunk(parsed, document_id=f"notebook-{extension}-debug")
    return {
        "path": path,
        "extension": extension,
        "parsed": parsed,
        "chunks": chunks,
        "stats": collect_stats(parsed, chunks),
    }


## Run The Current File

Re-run this cell after changing `selected_path`.


In [5]:
inspection = inspect_document(selected_path)
parsed = inspection["parsed"]
chunks = inspection["chunks"]
stats = inspection["stats"]

print(f"Parsed file: {inspection['path']}")
pprint(stats)


Parsed file: /Users/pelmeshek1706/Desktop/projects/knowledge_agent/data/Product_Requirements_Document_Personal_KB_v0.2.md
{'block_type_counts': {'markdown_section': 65},
 'chunk_char_avg': 594.5,
 'chunk_char_max': 1200,
 'chunk_char_min': 180,
 'chunk_count': 71,
 'chunker': 'MarkdownChunker',
 'file_extension': 'md',
 'file_path': 'data/Product_Requirements_Document_Personal_KB_v0.2.md',
 'parser': 'MarkdownParser',
 'raw_text_chars': 34901,
 'structured_block_count': 65,
 'unique_chunk_source_refs': 65}


## Parsed Metadata And Structured Blocks

This shows the parsed document metadata plus a compact preview of the first structured blocks.


In [6]:
metadata = parsed.metadata.model_dump(mode="json")
blocks_preview = [summarize_block(block) for block in parsed.structured_blocks[:10]]

print("Parsed metadata")
pprint(metadata)

print("\nStructured block preview")
pprint(blocks_preview)


Parsed metadata
{'created_at': None,
 'modified_at': '2026-05-12T21:32:06.581826Z',
 'size_bytes': 34959}

Structured block preview
[{'cell_range': None,
  'heading_path': ['Product Requirements Document — Personal KB Local-First '
                   'GraphRAG System'],
  'page_number': None,
  'sheet_name': None,
  'source_ref': {'file_path': 'data/Product_Requirements_Document_Personal_KB_v0.2.md',
                 'section': 'Product Requirements Document — Personal KB '
                            'Local-First GraphRAG System'},
  'text_preview': '**Status:** Draft v0.2 **Product name:** Personal KB '
                  '**Package name:** `personal_kb` **Related architecture:** '
                  '`Technical_Architecture_Personal_KB_v0.3.md` *...',
  'type': 'markdown_section'},
 {'cell_range': None,
  'heading_path': ['Product Requirements Document — Personal KB Local-First '
                   'GraphRAG System',
                   '1. Executive Summary'],
  'page_number': None,
 

## Chunks And Source Refs

This shows the first chunks, their source refs, and a de-duplicated preview of the source-ref set.


In [7]:
chunks_preview = [summarize_chunk(chunk) for chunk in chunks[:10]]
source_refs_preview = [chunk.source_ref.model_dump(mode="json") for chunk in chunks[:10]]
unique_source_refs = sorted(
    {
        json.dumps(chunk.source_ref.model_dump(mode="json"), sort_keys=True)
        for chunk in chunks
    }
)

print("Chunk preview")
pprint(chunks_preview)

print("\nSource refs for first chunks")
pprint(source_refs_preview)

print(f"\nUnique chunk source refs: {len(unique_source_refs)}")
pprint([json.loads(item) for item in unique_source_refs[:20]])


Chunk preview
[{'char_count': 437,
  'chunk_id': '0c3f6847-590f-5030-b962-66c2db4cb207',
  'chunk_index': 0,
  'source_ref': {'cell_range': None,
                 'file_path': 'data/Product_Requirements_Document_Personal_KB_v0.2.md',
                 'page': None,
                 'section': 'Product Requirements Document — Personal KB '
                            'Local-First GraphRAG System',
                 'sheet': None},
  'text_preview': 'Product Requirements Document — Personal KB Local-First '
                  'GraphRAG System **Status:** Draft v0.2 **Product name:** '
                  'Personal KB **Package name:** `personal_kb` ...'},
 {'char_count': 732,
  'chunk_id': 'd022e92c-dc7a-5658-8922-7a036681441c',
  'chunk_index': 1,
  'source_ref': {'cell_range': None,
                 'file_path': 'data/Product_Requirements_Document_Personal_KB_v0.2.md',
                 'page': None,
                 'section': 'Product Requirements Document — Personal KB '
                 